In [ ]:
'''
Both WL and NBI imaging technologies are included
Video clips were processed to extract the individual frames.
All frames were analysed and categorized into showing a polyp or not, as well as identifying the type of light source (WL or NBI)

'''

In [ ]:
import cv2
import pandas as pd
import numpy as np
import tkinter as tk
from tkinter import filedialog
from PIL import Image, ImageDraw
import os
from common import get_image_properties, get_image_size, resize_and_maintain_aspect, adjust_bounding_box, file_hash

In [ ]:
# Open file dialog to select the spreadsheet file
spreadsheet_path = filedialog.askopenfilename(
    title="Select the Spreadsheet File",
    filetypes=[("Excel files", "*.xlsx")]
)
piba_data = pd.read_excel(spreadsheet_path)
# Display the first few rows of the dataframe to understand its structure
piba_data.head()

In [ ]:
new_column_names = {
    'image_id': 'FRAME',
    'manually_selected': 'MANUAL SELECTION',
    'title': 'IMAGE TYPE',
    'size': 'SIZE',
    'paris_primary': 'SHAPE',
    'width': 'BBOX WIDTH',
    'height': 'BBOX HEIGHT',
    'x': 'X',
    'y': 'Y'
}
# Rename the columns
piba_data = piba_data.rename(columns=new_column_names)
piba_data.head()

In [ ]:
# Listing unique values in the 'IMAGE TYPE' column
unique_image_types = piba_data['IMAGE TYPE'].unique()

# Display the unique image type values
print(unique_image_types)

In [ ]:
# Replace values in 'IMAGE TYPE' with 'WL' or 'NBI'
piba_data['IMAGE TYPE'] = np.where(piba_data['IMAGE TYPE'].str.contains('WL'), 'WL',
                            np.where(piba_data['IMAGE TYPE'].str.contains('NBI'), 'NBI', piba_data['IMAGE TYPE']))
piba_data.head()

In [ ]:
# Listing unique values in the 'IMAGE TYPE' column
unique_sizes = piba_data['SIZE'].unique()

# Display the unique image type values
print(unique_sizes)

In [ ]:
na_rows = piba_data[piba_data['SIZE'].isna()]
print(na_rows)

In [ ]:
# Convert the float values in 'SIZE' to integers
piba_data['SIZE'] = piba_data['SIZE'].astype('Int64')
piba_data.head()

In [ ]:
unique_shape_values = piba_data['SHAPE'].unique()
print(unique_shape_values)

In [ ]:
# Add .png extension to each value in the 'FRAME' field
piba_data['FRAME'] = piba_data['FRAME'].apply(lambda x: f"{x}.png")

In [ ]:
na_rows = piba_data[piba_data['SHAPE'].isna()]
print(na_rows)

In [ ]:
piba_data = piba_data.dropna(subset=['SHAPE'])
na_rows = piba_data[piba_data['SHAPE'].isna()]
print(na_rows)

In [ ]:
# Dictionary mapping old text values to new codes
shape_codes = {
    'DEPRESSED': 'IIc',
    'FLAT': 'IIb',
    'PEDUNCULATED': 'Ip',
    'SESSILE': 'Is',
    'SLIGHTLY_ELEVATED': 'IIa',
    'ULCERATED': 'III'
}

# Replace the text values in the 'SHAPE' field with their corresponding codes
piba_data['SHAPE'] = piba_data['SHAPE'].replace(shape_codes)
unique_shape_values = piba_data['SHAPE'].unique()
print(unique_shape_values)

In [ ]:
base_path = os.path.dirname(spreadsheet_path)

for index, row in piba_data.iterrows():
    frame_id = row['FRAME']
    image_filename = f"{frame_id}"
    frame_path = os.path.join(base_path, 'media', image_filename)
    try:
        with Image.open(frame_path) as img:
            width, height = img.size
            piba_data.at[index, 'WIDTH'] = int(width)
            piba_data.at[index, 'HEIGHT'] = int(height)
            
    except IOError:
        print(f"Unable to open file: {frame_path}")

piba_data.head()

In [ ]:
unique_width_values = piba_data['WIDTH'].unique()
unique_height_values = piba_data['HEIGHT'].unique()
print(unique_width_values)
print(unique_height_values)

In [ ]:
def print_exceeding_bounding_boxes(df):
    # Define a lambda function to check if bounding box exceeds boundaries
    exceeds_boundaries = lambda row: (row['X'] + row['BBOX WIDTH'] > row['WIDTH']) or (row['Y'] + row['BBOX HEIGHT'] > row['HEIGHT'])
    
    # Apply the lambda function and filter rows where bounding box exceeds boundaries
    exceeding_rows = df[df.apply(exceeds_boundaries, axis=1)]
    
    # Check if any row exceeds boundaries
    if not exceeding_rows.empty:
        print("Rows where the bounding box exceeds image boundaries:")
        print(exceeding_rows)
    else:
        print("No bounding box exceeds image boundaries.")

# Call the function
print_exceeding_bounding_boxes(piba_data)

In [ ]:
# Step 1 & 2: Calculate the bottom-right corner coordinates
piba_data['X_BR'] = piba_data['X'] + piba_data['BBOX WIDTH']  # Bottom-right X coordinate
piba_data['Y_BR'] = piba_data['Y'] + piba_data['BBOX HEIGHT']  # Bottom-right Y coordinate

# Step 3: Combine values into a single array and store in new 'BOUNDING BOX' column
piba_data['BOUNDING BOX'] = piba_data.apply(lambda row: f"{row['X']},{row['Y']},{row['X'] + row['BBOX WIDTH']},{row['Y'] + row['BBOX HEIGHT']}", axis=1)

# Step 4: Drop the original bounding box columns
piba_data.drop(columns=['X', 'Y', 'BBOX WIDTH', 'BBOX HEIGHT', 'X_BR', 'Y_BR'], inplace=True)

piba_data.head()

In [ ]:
# Function to draw bounding boxes on an image
def draw_bounding_boxes(image_path, bounding_boxes, save_path):
    # Open the image
    with Image.open(image_path) as img:
        draw = ImageDraw.Draw(img)
        coordinates = [int(coord.strip()) for coord in bounding_boxes.split(',') if coord.strip()]
        # print(coordinates)
        draw.rectangle(coordinates, outline="red", width=2)
        
        # Display the image
        # plt.imshow(img)
        # plt.show()
        
        # Save the image with bounding boxes
        img.save(save_path)

save_base_path = os.path.join(base_path, 'img_bbox')

# Create the save directory if it doesn't exist
os.makedirs(save_base_path, exist_ok=True)

# Loop through the DataFrame and draw bounding boxes on each image
for index, row in piba_data.iterrows():
    frame_id = row['FRAME']
    image_filename = f"{frame_id}.png"
    frame_path = os.path.join(base_path, 'media', image_filename)
    save_path = os.path.join(save_base_path, f"bbox_{row['FRAME']}.png") 
    draw_bounding_boxes(frame_path, row['BOUNDING BOX'], save_path)


In [ ]:
piba_data.drop(columns=['MANUAL SELECTION'], inplace=True)

In [ ]:
# Saving the filtered DataFrame to a spreadsheet
save_path = os.path.join(base_path, 'piba-refined.xlsx')
piba_data.to_excel(save_path, index=False)

In [ ]:
# Database standardization for consolidation

In [ ]:
source_folder_base = os.path.join(base_path, 'media')
output_folder_base = os.path.join(base_path, 'img_refined')
os.makedirs(output_folder_base, exist_ok=True)
target_size = (854, 480)
piba_data.rename(columns={'FRAME': 'ORIGINAL FRAME'}, inplace=True)
piba_data['FRAME'] = None

In [ ]:
for index, row in piba_data.iterrows():
    original_path = os.path.join(source_folder_base, row['ORIGINAL FRAME'])
    original_size = (row['WIDTH'], row['HEIGHT'])
    
    # Use the resize function to adjust the image and maintain aspect ratio
    resized_image_path = os.path.join(output_folder_base, row['ORIGINAL FRAME'])  # Temp path for resizing logic
    resize_and_maintain_aspect(original_path, resized_image_path, target_size)
    
    # After resizing, calculate padding added for bounding box adjustment
    with Image.open(resized_image_path) as img:
        resized_size = img.size
        padding = ((target_size[0] - resized_size[0]) // 2, (target_size[1] - resized_size[1]) // 2)
    
    # Adjust the bounding box
    bounding_box = tuple(map(int, row['BOUNDING BOX'].split(',')))
    adjusted_box = adjust_bounding_box(bounding_box, original_size, resized_size, padding)

    # Update the DataFrame with adjusted bounding box
    piba_data.at[index, 'BOUNDING BOX'] = ','.join(map(str, adjusted_box))
    
    # Generate a new filename based on the file hash and update the DataFrame
    new_frame_name = f"{file_hash(resized_image_path)}.png"
    piba_data.at[index, 'FRAME'] = new_frame_name
    
    # Rename and move the resized image to the final destination with the new name
    final_path = os.path.join(output_folder_base, new_frame_name)
    os.rename(resized_image_path, final_path)
